# Stage 7 — Base ASAG Dependency Triple Matching

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 7 — Dependency Triple Matching  

---

### Purpose
This notebook demonstrates the **Stage 7 Component** for the Base ASAG system.

In previous stages, we implemented:
- **Stage 3:** Linguistic Preprocessing (tokenization, lemmatization, POS tagging).
- **Stage 4:** Dependency Parsing (syntactic heads, grammatical dependency labels).
- **Stage 5:** Dependency Triples (raw arcs and canonical `(subject, predicate, object)` relations).
- **Stage 6:** Lexical and Semantic Relationships (4-level hierarchy: exact, lemma, POS guard, WordNet path).

**Stage 7 establishes structural and semantic alignment between reference and student dependency relations.**

The matcher accomplishes four crucial research tasks:
1. **Role-Preserved Alignment:** Aligns `subject` $\leftrightarrow$ `subject`, `predicate` $\leftrightarrow$ `predicate`, and `object` $\leftrightarrow$ `object`. Relations are **never** treated as unordered bags of words.
2. **Interpretable Slot Evidence:** Calculates continuous triple scores via configurable slot weights ($w_s, w_p, w_o$), retaining full Stage 6 diagnostic evidence for every slot.
3. **One-to-One Deterministic Matching:** Enforces strict one-to-one alignment using a greedy selection algorithm with deterministic tie-breaking.
4. **Explicit Information Partitioning:** Divides all relations into three distinct sets:
   - **Matched Relations**
   - **Unmatched Reference Relations**
   - **Unmatched Student Relations**

> **CRITICAL RESEARCH BOUNDARIES:**  
> - Stage 7 does **not** classify unmatched student information (classification as correct additional, irrelevant, or incorrect belongs to Stage 13).
> - Stage 7 does **not** evaluate textual entailment (Stage 8).
> - Stage 7 does **not** compute student grades or sentence-level cosine similarities (Stage 9).
> - **UNMATCHED $\neq$ INCORRECT:** Unmatched relations simply represent information not aligned to the reference.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> Synthetic demonstration examples for pipeline verification only.

## 1. Imports and Setup

We import pandas for reporting and the full Base ASAG pipeline up to Stage 7:
- Stage 4: `DependencyParser`
- Stage 5: `DependencyTripleExtractor`
- Stage 6: `LexicalSemanticComparator`
- Stage 7: `TripleMatcher`, `MatchedRelation`, `MatchingResult`

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure repository root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from base_asag.dependency import DependencyParser, DependencyTripleExtractor, DependencyRelation
from base_asag.semantic import LexicalSemanticComparator
from base_asag.matching import TripleMatcher, MatchedRelation, MatchingResult

# Initialize pipeline components
parser = DependencyParser(model_name="en_core_web_sm")
extractor = DependencyTripleExtractor()
comparator = LexicalSemanticComparator()
matcher = TripleMatcher(
    comparator=comparator,
    subject_weight=1.0 / 3.0,
    predicate_weight=1.0 / 3.0,
    object_weight=1.0 / 3.0,
    alignment_threshold=0.5,
)

print("Initialized Stage 7 TripleMatcher successfully.")
print(f"Weights: Subject={matcher.subject_weight:.3f}, Predicate={matcher.predicate_weight:.3f}, Object={matcher.object_weight:.3f}")
print(f"Alignment Threshold: {matcher.alignment_threshold}")

Initialized Stage 7 TripleMatcher successfully.
Weights: Subject=0.333, Predicate=0.333, Object=0.333
Alignment Threshold: 0.5


## 2. Demonstration 1: Paraphrased Student Answer (Student A)

**Question:** *"What is the purpose of a primary key?"*  
- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student A:** *"A primary key uniquely identifies every row in a table."*  

We parse both sentences, extract canonical relations via Stage 5, and match them using Stage 7.

In [2]:
ref_doc = parser.parse("A primary key uniquely identifies each record in a table.")
stu_a_doc = parser.parse("A primary key uniquely identifies every row in a table.")

ref_triples = extractor.extract_document(ref_doc)
stu_a_triples = extractor.extract_document(stu_a_doc)

result_a = matcher.match(ref_triples, stu_a_triples)

print(f"Matched Relations Count:            {result_a.matched_count}")
print(f"Unmatched Reference Relations Count: {result_a.unmatched_reference_count}")
print(f"Unmatched Student Relations Count:   {result_a.unmatched_student_count}\n")

matched_a = result_a.matched_relations[0]
print(f"Reference Relation: {matched_a.reference_relation.to_tuple()}")
print(f"Student Relation:   {matched_a.student_relation.to_tuple()}")
print(f"Alignment Score:    {matched_a.triple_score:.6f}")
print(f"Match Method:       {matched_a.match_method}")

Matched Relations Count:            1
Unmatched Reference Relations Count: 0
Unmatched Student Relations Count:   0

Reference Relation: ('key', 'identifies', 'record')
Student Relation:   ('key', 'identifies', 'row')
Alignment Score:    0.708333
Match Method:       semantic_path_triple


### Slot-Level Diagnostic Evidence for Student A

Notice how each individual slot's evidence from Stage 6 is retained in the `slot_evidence` dictionary:

In [3]:
slot_rows = []
for slot_name, ev in matched_a.slot_evidence.items():
    slot_rows.append({
        "Slot": slot_name,
        "Ref Token": ev.source_text,
        "Student Token": ev.target_text,
        "Method": ev.method,
        "Score": round(ev.similarity_score, 4),
        "Relation": ev.semantic_relation,
        "Ref Synset": ev.synset_source,
        "Student Synset": ev.synset_target,
    })

pd.DataFrame(slot_rows)

,Slot,Ref Token,Student Token,Method,Score,Relation,Ref Synset,Student Synset
0,subject,key,key,exact_match,1.000,EXACT_MATCH,NaN,NaN
1,predicate,identifies,identifies,exact_match,1.000,EXACT_MATCH,NaN,NaN
2,object,record,row,wordnet_path,0.125,WORDNET_PATH,record.n.04,quarrel.n.01


## 3. Demonstration 2: Correct Answer + Additional Information (Student B)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student B:** *"A primary key uniquely identifies each record in a table and cannot contain NULL values."*  

Here, Student B provides the complete reference answer **plus** an additional clause (`cannot contain NULL values`).

**Key Research Rule:**  
Stage 7 matches the primary-key clause and leaves the conjoined clause as an **unmatched student relation**.  
Stage 7 does **not** classify it as correct, incorrect, or irrelevant.

In [4]:
stu_b_doc = parser.parse(
    "A primary key uniquely identifies each record in a table and cannot contain NULL values."
)
stu_b_triples = extractor.extract_document(stu_b_doc)

result_b = matcher.match(ref_triples, stu_b_triples)

print("STUDENT B MATCHING RESULT:")
print(f"  Matched Count:             {result_b.matched_count}")
print(f"  Unmatched Reference Count: {result_b.unmatched_reference_count}")
print(f"  Unmatched Student Count:   {result_b.unmatched_student_count}\n")

print("Matched Pairs:")
for m in result_b.matched_relations:
    print(f"  Ref: {m.reference_relation.to_tuple()} <--> Student: {m.student_relation.to_tuple()} (Score: {m.triple_score:.4f}, Method: {m.match_method})")

print("\nUnmatched Student Relations (Preserved for Stage 11-13):")
for u in result_b.unmatched_student_relations:
    print(f"  Student Unmatched: {u.to_tuple()} [Type: {u.relation_type}]")

STUDENT B MATCHING RESULT:
  Matched Count:             1
  Unmatched Reference Count: 0
  Unmatched Student Count:   1

Matched Pairs:
  Ref: ('key', 'identifies', 'record') <--> Student: ('key', 'identifies', 'record') (Score: 1.0000, Method: exact_triple)

Unmatched Student Relations (Preserved for Stage 11-13):
  Student Unmatched: ('key', 'contain', 'values') [Type: SPO]


## 4. Demonstration 3: Correct Answer + Irrelevant Information (Student D)

- **Reference Answer:** *"A primary key uniquely identifies each record in a table."*  
- **Student D:** *"A primary key uniquely identifies each record in a table. SQL was developed in the 1970s."*  

Student D provides the complete reference answer, but adds an unrelated historical fact.

**Behavior:**  
- The core relation `(key, identifies, record)` matches with score `1.0`.  
- The passive relation `('SQL', 'developed', None)` from the second sentence is preserved in `unmatched_student_relations`.  
- Stage 7 does **not** penalize or classify the SQL sentence.

In [5]:
stu_d_doc = parser.parse(
    "A primary key uniquely identifies each record in a table. SQL was developed in the 1970s."
)
stu_d_triples = extractor.extract_document(stu_d_doc)

result_d = matcher.match(ref_triples, stu_d_triples)

print("STUDENT D MATCHING RESULT:")
print(f"  Matched Count:             {result_d.matched_count}")
print(f"  Unmatched Reference Count: {result_d.unmatched_reference_count}")
print(f"  Unmatched Student Count:   {result_d.unmatched_student_count}\n")

print("Matched Pairs:")
for m in result_d.matched_relations:
    print(f"  Ref: {m.reference_relation.to_tuple()} <--> Student: {m.student_relation.to_tuple()} (Score: {m.triple_score:.4f})")

print("\nUnmatched Student Relations:")
for u in result_d.unmatched_student_relations:
    print(f"  Student Unmatched: {u.to_tuple()} [Type: {u.relation_type}]")

STUDENT D MATCHING RESULT:
  Matched Count:             1
  Unmatched Reference Count: 0
  Unmatched Student Count:   1

Matched Pairs:
  Ref: ('key', 'identifies', 'record') <--> Student: ('key', 'identifies', 'record') (Score: 1.0000)

Unmatched Student Relations:
  Student Unmatched: ('SQL', 'developed', None) [Type: PASSIVE_SP]


## 5. Preservation of Grammatical Roles (Not a Bag of Words)

A critical limitation of lexical bag-of-words or unconstrained keyword matchers is that they ignore syntactic directionality.

Consider an inverted sentence:
- **Reference:** *"A primary key identifies a record."* $\rightarrow$ `(key, identify, record)`
- **Inverted Student:** *"A record identifies a primary key."* $\rightarrow$ `(record, identify, key)`

In a bag-of-words or set-overlap approach, `{key, identify, record}` and `{record, identify, key}` have a **100% lexical match** ($1.0$).  
In Stage 7 triple matching, syntactic roles are strictly enforced:

In [6]:
ref_rel = DependencyRelation(
    subject="key", predicate="identifies", object="record",
    subject_lemma="key", predicate_lemma="identify", object_lemma="record",
    relation_type="SPO", source_dependency_relations=("nsubj", "dobj"),
    subject_index=0, predicate_index=1, object_index=2
)

stu_inverted = DependencyRelation(
    subject="record", predicate="identifies", object="key",
    subject_lemma="record", predicate_lemma="identify", object_lemma="key",
    relation_type="SPO", source_dependency_relations=("nsubj", "dobj"),
    subject_index=0, predicate_index=1, object_index=2
)

# Test with strict threshold 0.6
strict_matcher = TripleMatcher(alignment_threshold=0.6)
result_inverted = strict_matcher.match([ref_rel], [stu_inverted])

cand = result_inverted.candidate_alignments[0]
print(f"Candidate Pair: {ref_rel.to_tuple()} vs {stu_inverted.to_tuple()}")
print(f"  Subject Slot (key <-> record):   {cand.subject_similarity:.4f}")
print(f"  Predicate Slot (identifies <-> identifies): {cand.predicate_similarity:.4f}")
print(f"  Object Slot (record <-> key):    {cand.object_similarity:.4f}")
print(f"  Triple Alignment Score:          {cand.triple_score:.4f}")
print(f"  Meets Threshold (0.6)?           {cand.is_eligible}")
print(f"  Result Matched Count:            {result_inverted.matched_count}")

Candidate Pair: ('key', 'identifies', 'record') vs ('record', 'identifies', 'key')
  Subject Slot (key <-> record):   0.3333
  Predicate Slot (identifies <-> identifies): 1.0000
  Object Slot (record <-> key):    0.3333
  Triple Alignment Score:          0.5556
  Meets Threshold (0.6)?           False
  Result Matched Count:            0


## 6. Intransitive (SP) Relations & Active Weight Normalization

When both reference and student relations are intransitive clauses lacking direct objects (e.g. *"The system terminates"*), Stage 7 normalizes weights over the active slots (subject + predicate) rather than artificially inserting a zero-valued object penalty.

In [7]:
sp_ref = DependencyRelation(
    subject="system", predicate="terminates", object=None,
    subject_lemma="system", predicate_lemma="terminate", object_lemma=None,
    relation_type="SP", source_dependency_relations=("nsubj",),
    subject_index=0, predicate_index=1, object_index=None
)

sp_stu = DependencyRelation(
    subject="system", predicate="terminates", object=None,
    subject_lemma="system", predicate_lemma="terminate", object_lemma=None,
    relation_type="SP", source_dependency_relations=("nsubj",),
    subject_index=0, predicate_index=1, object_index=None
)

result_sp = matcher.match([sp_ref], [sp_stu])
matched_sp = result_sp.matched_relations[0]

print(f"Reference SP: {matched_sp.reference_relation.to_tuple()}")
print(f"Student SP:   {matched_sp.student_relation.to_tuple()}")
print(f"Triple Score: {matched_sp.triple_score:.4f}")
print(f"Object Sim:   {matched_sp.object_similarity} (Properly handled as None)")
print(f"Match Method: {matched_sp.match_method}")

Reference SP: ('system', 'terminates', None)
Student SP:   ('system', 'terminates', None)
Triple Score: 1.0000
Object Sim:   None (Properly handled as None)
Match Method: exact_triple


## 7. Comparative Summary Matrix

We summarize the matching results across the demonstration cases to observe the behavior of the alignment engine.

In [8]:
summary_data = [
    {
        "Student": "Student A (Paraphrased: row)",
        "Matched Pairs": result_a.matched_count,
        "Unmatched Ref": result_a.unmatched_reference_count,
        "Unmatched Stu": result_a.unmatched_student_count,
        "Top Score": round(result_a.matched_relations[0].triple_score, 4) if result_a.matched_count > 0 else 0.0,
        "Match Method": result_a.matched_relations[0].match_method if result_a.matched_count > 0 else "None",
    },
    {
        "Student": "Student B (Extra info: NULL values)",
        "Matched Pairs": result_b.matched_count,
        "Unmatched Ref": result_b.unmatched_reference_count,
        "Unmatched Stu": result_b.unmatched_student_count,
        "Top Score": round(result_b.matched_relations[0].triple_score, 4) if result_b.matched_count > 0 else 0.0,
        "Match Method": result_b.matched_relations[0].match_method if result_b.matched_count > 0 else "None",
    },
    {
        "Student": "Student D (Irrelevant: SQL history)",
        "Matched Pairs": result_d.matched_count,
        "Unmatched Ref": result_d.unmatched_reference_count,
        "Unmatched Stu": result_d.unmatched_student_count,
        "Top Score": round(result_d.matched_relations[0].triple_score, 4) if result_d.matched_count > 0 else 0.0,
        "Match Method": result_d.matched_relations[0].match_method if result_d.matched_count > 0 else "None",
    },
    {
        "Student": "Inverted Roles (record identifies key)",
        "Matched Pairs": result_inverted.matched_count,
        "Unmatched Ref": result_inverted.unmatched_reference_count,
        "Unmatched Stu": result_inverted.unmatched_student_count,
        "Top Score": round(cand.triple_score, 4),
        "Match Method": "Rejected (below 0.6 threshold)",
    },
]

pd.DataFrame(summary_data)

,Student,Matched Pairs,Unmatched Ref,Unmatched Stu,Top Score,Match Method
0,Student A (Paraphrased: row),1,0,0,0.7083,semantic_path_triple
1,Student B (Extra info: NULL values),1,0,1,1.0000,exact_triple
2,Student D (Irrelevant: SQL history),1,0,1,1.0000,exact_triple
3,Inverted Roles (record identifies key),0,1,1,0.5556,Rejected (below 0.6 threshold)


## 8. Research Boundaries & Conclusion

### What Stage 7 Achieves
1. **Structural & Semantic Coupling:** Combines grammatical structure from Stage 5 with multi-tiered lexical semantics from Stage 6.
2. **Grammatical Role Soundness:** Preserves subject, predicate, and object roles.
3. **One-to-One Alignment:** Deterministic greedy matching with explicit tie-breaking prevents duplicate alignments.
4. **Preservation of Unmatched Information:** Unmatched student relations are explicitly preserved without premature classification.

### Boundaries Maintained
- Stage 7 does **not** perform whole-sentence embeddings, TF-IDF, or cosine similarity.
- Stage 7 does **not** perform textual entailment or context relevance filtering (Stage 8).
- Stage 7 does **not** compute a final ASAG grade (Stage 9).
- Stage 7 does **not** classify unmatched information as correct/incorrect/irrelevant (Stage 13).